# KIRA — 2B. Préentraînement (GPU T4 ×2)

KIRA LM apprend le français à partir de **poids initialisés de zéro**. Une session dure au plus ~11 h et s'arrête
d'elle-même ; la suivante reprend exactement à la même étape.

Kaggle ne laisse pas un notebook relire sa propre sortie : on **alterne** entre deux notebooks identiques,
**2A** et **2B**. Chacun prend en entrée la sortie de l'autre : A → B → A → B…

**Réglages Kaggle** : *Accelerator* = **GPU T4 x2** ; *Internet* = **On** ; *Environment* = **Pin to original environment**.
**Add Input** : la sortie de **kira-1-donnees**, et la sortie de **kira-2A** (sauf pour la toute première session).
Avant de lancer, vérifie dans *Input* que l'entrée kira-2A est à sa **dernière version**.
Puis **Save Version → Save & Run All (Commit)**.

Première session du run : `NEW_RUN = True` (puis remets `False`). Pour une simple mesure de vitesse, utilise
un autre `RUN_NAME` avec `MEASURE_STEPS = 30`.
Fin du parcours : fixe `DECAY_START` (l'étape actuelle) et `DECAY_STEPS` (par exemple 15 % des étapes faites) ;
le taux d'apprentissage descend alors jusqu'à zéro et le run se termine.

In [ ]:
# Paramètres
RUN_NAME = "kira-100m-a"   # même nom pour toutes les sessions d'un même run
NEW_RUN = False            # True seulement pour la toute première session de ce run
RESUME_STEP = None         # facultatif : étape attendue (affichée à la fin de la session précédente)
PRESET = "kira-100m"       # kira-30m, kira-100m ou kira-125m-deep
BATCH_TOKENS = 524288      # jetons par mise à jour (identique d'une session à l'autre)
MICRO_BATCH = 8            # séquences par GPU et par passe ; 4 si la mémoire manque, 16 si elle suffit
LR = 0.0015               # fixé pour tout le run (float16 sur T4 : prudence)
WARMUP = 500
EVAL_EVERY = 500           # mesure de validation et textes générés toutes les N étapes
MIX = ""                   # vide = poids du manifeste ; sinon "source=poids,source=poids"
TIME_LIMIT_HOURS = 11.0    # durée d'entraînement de cette session
MEASURE_STEPS = None       # ex. 30 pour une simple mesure de vitesse
DECAY_START = None         # ex. 9000 : début de la décroissance finale
DECAY_STEPS = None         # ex. 1500
KIRA_REF = None            # None = même commit que la session précédente (ou "main" pour un nouveau run)
ALLOW_CODE_CHANGE = False  # True seulement après avoir lu ce qui a changé dans le code
NCCL_P2P_DISABLE = False   # True seulement si l'entraînement à deux GPU reste bloqué au démarrage

In [ ]:
# Retrouve les entrées ajoutées au notebook (« Add Input »), en lecture seule sous /kaggle/input.
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_data():
    found = []
    for manifest in Path("/kaggle/input").rglob("manifest.json"):
        try:
            if json.loads(manifest.read_text(encoding="utf-8")).get("format") == "kira-token-shards-1":
                found.append(manifest.parent)
        except (OSError, ValueError):
            pass
    if len(found) != 1:
        raise SystemExit("Il faut exactement une entrée de données KIRA (sortie du notebook kira-1-donnees). "
                         f"Trouvé : {[str(f) for f in found]}")
    return found[0]

def find_run(name):
    """Session la plus avancée d'un run parmi les entrées : (dossier, étape), ou (None, None)."""
    best, best_step = None, None
    for state in Path("/kaggle/input").rglob(f"runs/{name}/training.pt"):
        manifest = state.parent / "manifest.json"
        step = json.loads(manifest.read_text(encoding="utf-8")).get("steps", 0) if manifest.exists() else 0
        if best_step is None or step > best_step:
            best, best_step = state.parent, step
    return best, best_step

In [ ]:
DATA = find_data()
PREVIOUS, PREVIOUS_STEP = find_run(RUN_NAME)
if PREVIOUS is None and not NEW_RUN:
    raise SystemExit(f"Aucune session précédente de {RUN_NAME} dans les entrées. Première session : NEW_RUN = True. "
                     "Sinon, ajouter en entrée la sortie de l'autre notebook (2A ↔ 2B) à sa dernière version.")
if PREVIOUS is not None and NEW_RUN:
    raise SystemExit(f"NEW_RUN = True alors que {RUN_NAME} existe déjà (étape {PREVIOUS_STEP}) : remettre NEW_RUN = False "
                     "pour continuer, ou choisir un autre RUN_NAME.")
if RESUME_STEP is not None and PREVIOUS_STEP != RESUME_STEP:
    raise SystemExit(f"Étape {PREVIOUS_STEP} trouvée au lieu de {RESUME_STEP} : mettre l'entrée à sa dernière version.")
if KIRA_REF is None:
    KIRA_REF = "main"
    if PREVIOUS is not None:
        KIRA_REF = json.loads((PREVIOUS / "manifest.json").read_text(encoding="utf-8"))["sessions"][-1].get("code_commit") or "main"
print("Données :", DATA)
print("Reprise :", f"{PREVIOUS} (étape {PREVIOUS_STEP})" if PREVIOUS else "aucune — nouveau run, poids aléatoires")

In [ ]:
# Récupère le code public de KIRA au commit choisi et installe le tokenizer épinglé.
def run(*command, **options):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **options)

CODE = Path("/tmp/KIRA")
if not (CODE / ".git").exists():
    run("git", "clone", "--quiet", "https://github.com/BlackGhost99/KIRA", CODE)
run("git", "-C", CODE, "fetch", "--quiet", "origin")
run("git", "-C", CODE, "checkout", "--quiet", "--detach", KIRA_REF if len(KIRA_REF) == 40 else "origin/" + KIRA_REF)
COMMIT = subprocess.run(["git", "-C", str(CODE), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
os.environ.update(KIRA_CODE_COMMIT=COMMIT, PYTHONPATH=str(CODE), PYTHONUNBUFFERED="1")
run(sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "tokenizers==0.23.3")
import torch
image = Path("/etc/git_commit")
print("Code KIRA :", COMMIT, "| PyTorch", torch.__version__, "| GPU :", torch.cuda.device_count(),
      "| image Kaggle :", image.read_text().strip() if image.exists() else "?")

In [ ]:
# Lance une commande et garde aussi sa sortie dans un fichier journal.
def stream(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", " ".join(map(str, command)), flush=True)
    with open(log_path, "a", encoding="utf-8") as log:
        process = subprocess.Popen([str(c) for c in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise SystemExit(f"Échec (code {code}) : voir {log_path}. Les sauvegardes déjà écrites restent dans /kaggle/working.")

In [ ]:
GPUS = torch.cuda.device_count()
if not GPUS:
    raise SystemExit("Aucun GPU visible : choisir Accelerator = GPU T4 x2 dans les réglages du notebook.")
run("nvidia-smi")
OUTPUT = Path("/kaggle/working/runs") / RUN_NAME
arguments = ["pretrain", "--data", DATA, "--output", OUTPUT, "--preset", PRESET, "--batch-tokens", BATCH_TOKENS,
             "--micro-batch", MICRO_BATCH, "--lr", LR, "--warmup", WARMUP, "--eval-every", EVAL_EVERY,
             "--time-limit", int(TIME_LIMIT_HOURS * 3600)]
if PREVIOUS is not None:
    arguments += ["--resume-from", PREVIOUS]
if MIX:
    arguments += ["--mix", MIX]
if MEASURE_STEPS:
    arguments += ["--session-steps", MEASURE_STEPS]
if DECAY_START is not None:
    arguments += ["--decay-start", DECAY_START, "--decay-steps", DECAY_STEPS]
if ALLOW_CODE_CHANGE:
    arguments += ["--allow-code-change"]
if NCCL_P2P_DISABLE:
    os.environ["NCCL_P2P_DISABLE"] = "1"
launcher = ([sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", GPUS]
            if GPUS > 1 else [sys.executable])
# Journal à part : le dossier du run ne doit contenir que ce que l'entraîneur écrit.
stream([*launcher, "-m", "research.kira_model.gpu_train", *arguments], Path("/kaggle/working/journaux") / f"{RUN_NAME}.txt")

In [ ]:
# Résumé de la session : vitesse, pertes de validation et textes générés (à relire : ce n'est pas une note de qualité).
manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
session = manifest["sessions"][-1]
speeds = [e["tokens_per_second"] for e in manifest["history"] if "tokens_per_second" in e][-20:]
print(f"Étape {manifest['steps']} | {manifest['tokens_seen']/1e9:.3f} G jetons vus | arrêt : {session['stop_reason']}")
print(f"➜ Prochaine session : l'autre notebook, avec RESUME_STEP = {manifest['steps']}")
if speeds:
    rate = sum(speeds) / len(speeds)
    print(f"Vitesse : {rate:,.0f} jetons/s, soit ~{rate * 3600 / 1e9:.2f} G jetons par heure de GPU")
for name, value in manifest["last_eval"]["language"].items():
    print(f"Validation {name:24} perte {value['loss_per_token']:.4f}")
for sample in manifest["samples"][-1]["samples"] or []:
    print("\n«", sample["prompt"], "» →", sample["sampled"][:300])